# Sim Flip Chip Distance - Route B - Mesh Profile With Distance Field

This notebook profiles the same big Route B XAO with the SGB diagnostic
Mesh Gate's gsim-like Distance/Threshold field. It demonstrates why the
large geometry should be tested with the no-field baseline first.

Observed result on the saved probe:

| Stage | Time | Output |
| --- | ---: | --- |
| open XAO | `3.43s` | XAO opened successfully |
| physical groups | `0.23s` | `127/127` matched |
| Distance field setup | `0.20s` | `154019` boundary curves |
| `generate(1)` | `>5 min` | interrupted |

The setup itself is cheap; the expensive part is evaluating the 154k-curve
Distance field during 1D mesh generation.


In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file():
    if ROOT.parent == ROOT:
        raise RuntimeError("Could not locate repository root")
    ROOT = ROOT.parent

RUN_FOLDER = ROOT / "tutorials" / "runs" / "no_inset" / "sim_flip_chip_distance" / "route_b_ignore_port_sheet_probe"
METADATA_DIR = RUN_FOLDER / "metadata" / "semantic_geometry"
XAO_PATH = RUN_FOLDER / "geometry" / "semantic_geometry_route_b.xao"
PHYSICAL_GROUPS_PATH = METADATA_DIR / "04_export_physical_groups.json"

print("run folder:", RUN_FOLDER)
print("XAO exists:", XAO_PATH.is_file(), XAO_PATH)
print("physical groups exists:", PHYSICAL_GROUPS_PATH.is_file(), PHYSICAL_GROUPS_PATH)


In [ ]:
def read_jsonl(path):
    rows = []
    if not path.is_file():
        print("missing", path)
        return rows
    for line in path.read_text().splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def print_profile(rows):
    for row in rows:
        event = row.get("event")
        seconds = row.get("seconds")
        elapsed = row.get("elapsed_s")
        detail = row.get("counts") or row.get("refinement") or row.get("entities") or ""
        print(f"{event:24s} seconds={seconds!s:>10s} elapsed={elapsed!s:>10s} {detail}")

rows = read_jsonl(METADATA_DIR / "mesh_stage_profile_gsim_default_distance_1d.jsonl")
print_profile(rows)


## Optional Rerun

This profile may run for many minutes inside `generate(1)`. Keep
`RUN_PROFILE = False` unless you are intentionally reproducing the
bottleneck.


In [ ]:
RUN_PROFILE = False

if not RUN_PROFILE:
    print("Set RUN_PROFILE = True to rerun the Distance-field 1D profile.")
else:
    import time
    import gmsh
    from semantic_geometry_builder.mesh_gate import (
        _apply_mesh_profile,
        _check_physical_groups,
        _load_physical_group_records,
        _setup_gsim_like_refinement,
    )

    failures = []
    records = _load_physical_group_records(PHYSICAL_GROUPS_PATH, failures)
    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 0)
    try:
        gmsh.open(str(XAO_PATH))
        _apply_mesh_profile(gmsh, "gsim_default")
        group_check = _check_physical_groups(gmsh, records, failures)
        refinement = _setup_gsim_like_refinement(
            gmsh,
            group_check["surface_entity_tags"],
            refined_mesh_size_um=5.0,
            max_mesh_size_um=300.0,
        )
        print("refinement:", {k: v for k, v in refinement.items() if k != "field_ids"})
        started = time.perf_counter()
        gmsh.model.mesh.generate(1)
        print("generate(1)", round(time.perf_counter() - started, 3))
    finally:
        gmsh.clear()
        gmsh.finalize()
